# 0단계: 공개된 Octopus v2 관찰하기
목표: 학습된 완성품을 뜯어보고, 논문의 주장이 **코드/텐서 수준에서 어떻게 보이는지** 확인한다.

진행 규칙
- 각 과제마다 **실행 전에 예측**을 먼저 적는다.
- 코드는 직접 작성한다. 막히면 Claude에게 "과제 N 힌트 1" 처럼 요청 (힌트 1: 개념 → 2: API 이름 → 3: 코드 조각).
- 런타임: **T4 GPU**, 🔑 Secrets에 `HF_TOKEN` 등록.

## 준비 (보일러플레이트 — 그냥 실행)

In [6]:
!pip -q install -U transformers accelerate
from huggingface_hub import login
from google.colab import userdata

# 코랩 왼쪽 열쇠 아이콘(Secrets)에 'HF_TOKEN' 이름으로 토큰을 등록한 뒤 실행하세요.
try:
    login(userdata.get("HF_TOKEN"))
except Exception as e:
    print("Secrets에 'HF_TOKEN'이 등록되어 있지 않거나 오류가 발생했습니다.")

---
## 과제 1. 모델과 토크나이저 로드
`NexaAIDev/Octopus-v2` 를 GPU에 올린다.
- 조건: T4는 bfloat16을 지원하지 않는다. 어떤 dtype을 써야 할까?
- 확인: 모델 아키텍처 이름, vocab 크기 출력

**예측**: 이 모델의 베이스는 무엇이고, vocab 크기는 원래 베이스 모델보다 클까?
> (여기에 예측)

In [8]:
# 과제 1
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "NexaAIDev/Octopus-v2"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto",
    )
model.eval() # 학습용으로 키면 dropout 생김.

print(model.config.architectures)
print(model.config.vocab_size) # original gemma와 비교해서 토큰이 22개 늘었다. 그게 특수토큰인거임!!


Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

['GemmaForCausalLM']
256022


---
## 과제 2. 추가된 특수 토큰 찾기
- `<nexa`로 시작하는 토큰을 전부 찾아 (토큰, ID) 목록을 출력
- 개수, ID 범위 확인
- `len(tokenizer)`, input embedding 행렬 shape, lm_head shape 비교
- `"<nexa_3>"`과 `"take_a_photo"`를 각각 토크나이즈하면 토큰이 몇 개로 쪼개지는지

**예측**: nexa 토큰은 몇 개? ID는 vocab 맨 끝에 붙어 있을까?
> (여기에 예측)

In [15]:
# 과제 2
vocab = tokenizer.get_vocab()

nexa = {}
for token, idx in vocab.items():
  if token.startswith("<nexa"):
    nexa[token] = idx

print(len(nexa))
for token, idx in sorted(nexa.items(), key=lambda x: x[1]):   # ID 순으로 정렬
    print(idx, token)
print(model.get_input_embeddings().weight.shape)
print(model.get_output_embeddings().weight.shape)

for s in ["<nexa_3>", "<nexa_end>", "take_a_photo"]:
    ids = tokenizer(s, add_special_tokens=False).input_ids
    print(s, "->", ids, "->", tokenizer.convert_ids_to_tokens(ids))

22
256000 <nexa_0>
256001 <nexa_1>
256002 <nexa_2>
256003 <nexa_3>
256004 <nexa_4>
256005 <nexa_5>
256006 <nexa_6>
256007 <nexa_7>
256008 <nexa_8>
256009 <nexa_9>
256010 <nexa_10>
256011 <nexa_11>
256012 <nexa_12>
256013 <nexa_13>
256014 <nexa_14>
256015 <nexa_15>
256016 <nexa_16>
256017 <nexa_17>
256018 <nexa_18>
256019 <nexa_19>
256020 <nexa_20>
256021 <nexa_end>
torch.Size([256022, 2048])
torch.Size([256022, 2048])
<nexa_3> -> [256003] -> ['<nexa_3>']
<nexa_end> -> [256021] -> ['<nexa_end>']
take_a_photo -> [9399, 235298, 235250, 235298, 5817] -> ['take', '_', 'a', '_', 'photo']


넥사 토큰들이 더이상 나눠지지 않는 최소 단위임을 확인했다.  
넥사 토큰들은 마지막에 붙어있다.

---
## 과제 3. 추론 함수 만들기
`octopus(query) -> (생성 텍스트, 생성 토큰 수, 걸린 시간)` 을 구현한다.
- 프롬프트 템플릿은 README의 논문 포맷 참고 (모델 카드에는 `Query: ...

Response:` 형태)
- greedy decoding
- 특수 토큰이 보이도록 디코딩해야 한다
- 테스트 질의: 셀카 찍기 / 알람 설정 / 유튜브 검색 / **함수와 무관한 질문 1개**

**예측**: 출력 형식은? 함수 설명(Function description)까지 생성될까? 무관한 질문엔 뭐라고 할까?
> (여기에 예측)

In [ ]:
# 과제 3


**질문**
1. 생성이 `<nexa_end>`에서 멈췄는가? 그 이유를 `generation_config`에서 근거를 찾아 설명하라.
2. 무관한 질문에 대한 출력은 학습 데이터의 어떤 부분에서 온 행동일까?
> (여기에 답)

In [ ]:
# 질문 1 확인용


---
## 과제 4. "함수 선택 = 토큰 1개 분류" 검증
논문의 핵심 주장: 함수 이름 생성이 N개 functional token 중 argmax 하나로 끝난다.
- `Response:` 직후 **첫 생성 위치의 확률 분포**를 구한다 (generate 말고 forward 한 번)
- top-8 토큰과 확률 출력
- 모든 `<nexa_i>` (end 제외) 확률의 합 출력
- 함수 질의 1개, 무관 질의 1개 비교

**예측**: 함수 질의에서 1등 토큰 확률은 대략 얼마일까?
> (여기에 예측)

In [ ]:
# 과제 4


**질문**: 같은 함수를 기존 방식(함수 이름을 일반 텍스트로 생성)으로 고르면, 몇 토큰을 맞혀야 하나? 그게 정확도/속도에 왜 불리한가?
> (여기에 답)

---
## 정리 (3~5줄, 내 말로)
-
-
- 1~3단계 내 구현에 반영할 점: